In [4]:
import boto3
s3_client = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    region_name="ru-central1",
    aws_access_key_id="YCAJEK6EmbclThg_wry3OwCsF",
    aws_secret_access_key="YCORGYliOZhkb3VJRLOdQDxBuFNV_46YsjZZJ3XX"
)


In [9]:
import boto3
from PIL import Image

# Assuming AWS credentials are set up (e.g., via environment variables or IAM)

bucket_name = 'sneakers-hse-images-test'  # Replace with your actual bucket name

# Get list of folders (labels) in the bucket
response = s3_client.list_objects_v2(
    Bucket=bucket_name,
    Prefix='yolo_preprocessed/search-dataset-images/',
    Delimiter='/'
)
folders = [prefix['Prefix'].rstrip('/') for prefix in response.get('CommonPrefixes', [])]


In [10]:
folders

['yolo_preprocessed/search-dataset-images/Kari Кеды',
 'yolo_preprocessed/search-dataset-images/Kari Кроссовки',
 'yolo_preprocessed/search-dataset-images/Maison David Кеды',
 'yolo_preprocessed/search-dataset-images/Maison David Кроссовки',
 'yolo_preprocessed/search-dataset-images/Nike Кеды Dunk Low Retro',
 'yolo_preprocessed/search-dataset-images/Nike Кеды FIELD GENERAL',
 'yolo_preprocessed/search-dataset-images/Nike Кеды NIKE AIR FORCE 1',
 'yolo_preprocessed/search-dataset-images/Nike Кроссовки AIR MAX 90',
 'yolo_preprocessed/search-dataset-images/Nike Кроссовки AIR PEGASUS 2005',
 'yolo_preprocessed/search-dataset-images/Nike Кроссовки Nike Zoom Vomero 5',
 'yolo_preprocessed/search-dataset-images/PUMA Кеды CA Pro Classic II',
 'yolo_preprocessed/search-dataset-images/PUMA Кеды Palermo',
 'yolo_preprocessed/search-dataset-images/PUMA Кроссовки Darter Pro',
 'yolo_preprocessed/search-dataset-images/PUMA Кроссовки Hypnotic LS',
 'yolo_preprocessed/search-dataset-images/PUMA Крос

In [11]:
my_samples_list = []
for label in folders:
    # List objects in the folder
    # рекурсивно читаем все файлы внутри label, включая вложенные папки
    paginator = s3_client.get_paginator('list_objects_v2')
    objects = {'Contents': []}
    for page in paginator.paginate(Bucket=bucket_name, Prefix=f'{label}/'):
        objects['Contents'].extend(page.get('Contents', []))
    for obj in objects.get('Contents', []):
        if obj['Key'].endswith(('.jpg', '.png', '.jpeg')):  # Assuming image extensions
            my_samples_list.append({
                'path': f's3://{bucket_name}/{obj["Key"]}',
                'label': label
            })

In [ ]:
import os


os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = "ru-central1"
os.environ["AWS_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
print("✓ Переменные окружения установлены")

✓ Переменные окружения установлены


In [24]:
import PIL


In [26]:
my_samples_list[:5]

[{'path': 's3://sneakers-hse-images-test/yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_1.jpeg',
  'label': 'yolo_preprocessed/search-dataset-images/Kari Кеды'},
 {'path': 's3://sneakers-hse-images-test/yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_10.jpeg',
  'label': 'yolo_preprocessed/search-dataset-images/Kari Кеды'},
 {'path': 's3://sneakers-hse-images-test/yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_100.jpeg',
  'label': 'yolo_preprocessed/search-dataset-images/Kari Кеды'},
 {'path': 's3://sneakers-hse-images-test/yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_101.jpeg',
  'label': 'yolo_preprocessed/search-dataset-images/Kari Кеды'},
 {'path': 's3://sneakers-hse-images-test/yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_102.jpeg',
  'label': 'yolo_preprocessed/search-dataset-images/Kari Кеды'}]

In [27]:
from litdata import optimize
from io import BytesIO

def encode(sample):
    response = s3_client.get_object(Bucket=bucket_name, Key=sample["path"].replace(f"s3://{bucket_name}/", ""))
    image_data = response['Body'].read()
    return {
        "image": PIL.Image.open(BytesIO(image_data)),
        "label": sample["label"],
    }

# Now optimize the dataset
optimize(
    fn=encode,
    inputs=my_samples_list,
    output_dir="s3://sneakers-hse-images-test/yolo_preprocessed/train",  # Adjust output_dir as needed
    chunk_bytes="64MB",
    num_workers=8,
)

Create an account on https://lightning.ai/ to optimize your data faster using multiple nodes and large machines.
Setting multiprocessing start_method to fork. Tip: Libraries relying on lock can hang with `fork`. To use `spawn` in notebooks, move your code to files and import it within the notebook.
Storing the files under s3://sneakers-hse-images-test/yolo_preprocessed/train
Setup started with fast_dev_run=False.
Setup finished in 0.003 seconds. Found 11265 items to process.
Starting 8 workers with 11265 items. The progress bar is only updated when a worker finishes.
Workers are ready ! Starting data processing...


Progress:   0%|          | 0/11265 [00:00<?, ?it/s]

Rank 3 inferred the following `['jpeg', 'str']` data format.
Rank 7 inferred the following `['jpeg', 'str']` data format.
Rank 0 inferred the following `['jpeg', 'str']` data format.
Rank 6 inferred the following `['jpeg', 'str']` data format.
Rank 4 inferred the following `['jpeg', 'str']` data format.
Rank 1 inferred the following `['jpeg', 'str']` data format.
Rank 5 inferred the following `['jpeg', 'str']` data format.
Rank 2 inferred the following `['jpeg', 'str']` data format.
Worker 0 is terminating.
Worker 0 is done.
Worker 6 is terminating.
Worker 6 is done.
Worker 4 is terminating.
Worker 4 is done.
Worker 3 is terminating.
Worker 3 is done.
Worker 5 is terminating.
Worker 7 is terminating.
Worker 5 is done.
Worker 7 is done.
Worker 2 is terminating.
Worker 1 is terminating.
Worker 2 is done.
Worker 1 is done.
Workers are finished.
Finished data processing!
